# 09: 여러 학습 시드의 문장 결합 증강 비교

학습 시드 43, 44, 45에 대해 증강 on 학습과 평가, off 학습과 평가를 같은 Jupyter 커널에서 순차 실행합니다. 전체 셀 실행으로 6회 학습과 6회 평가 및 비교 집계까지 진행합니다. 프로젝트 `.venv/Scripts/python.exe` 커널을 선택합니다.

분할 시드 42, 학습 1,568건과 평가 555건을 고정합니다. 07과 08의 코드를 이 노트북 내부에 독립적으로 구성했으며 실행 중 다른 실험의 파일을 읽지 않습니다. 시드 42 결과는 집계에 포함하지 않습니다. 기존 결과를 덮어쓰거나 실패한 학습을 자동 재개하지 않습니다.


## 1. 실험과 학습 및 평가 파라미터


In [1]:
EXPERIMENT_PARAMS = {
    "experiment_dir": "experiments/09",
    "seeds": [43, 44, 45],
    "augmentation_modes": {"on": 0.5, "off": 0.0},
    "summary_file": "experiments/09/data/result/comparison.json",
    "expected_train_count": 1568,
    "expected_test_count": 555,
    "split_seed": 42,
    "excluded_cluster": 22,
}
TRAIN_RUNTIME_TEMPLATE = {'prefix_path': 'experiments/09/data',
 'model_name': 'Qwen/Qwen3-8B',
 'train_file': 'train.jsonl',
 'test_file': 'test.jsonl',
 'output_dir': 'models/checkpoints',
 'adapter_path': 'models',
 'run_config_file': 'result/run_config.json',
 'training_history_file': 'result/training_history.json',
 'report_file': 'result/evaluation.json',
 'device': 'cuda',
 'model_revision': 'b968826d9c46dd6066d109eabc6255188de91218'}

TRAIN_HYPER_TEMPLATE = {'augmentation': {'probability': 0.0,
                  'max_concat': 2,
                  'delimiters': [' ', '  ', ' / ', ' . ', '\n', ' , ', ' + ']},
 'chat_template': {'enable_thinking': False},
 'training': {'epochs': 3,
              'batch_size': 1,
              'grad_accum': 8,
              'learning_rate': 0.0002,
              'max_seq_length': 1280,
              'seed': 43},
 'lora': {'r': 16,
          'lora_alpha': 32,
          'lora_dropout': 0.05,
          'task_type': 'CAUSAL_LM',
          'bias': 'none',
          'target_modules': ['q_proj',
                             'k_proj',
                             'v_proj',
                             'o_proj',
                             'gate_proj',
                             'up_proj',
                             'down_proj']},
 'quantization': {'load_in_4bit': True,
                  'bnb_4bit_quant_type': 'nf4',
                  'bnb_4bit_use_double_quant': True},
 'precision': {'compute_dtype': 'auto', 'frozen_embedding_dtype': 'auto'},
 'model_loading': {'device_map': {'': 0},
                   'attn_implementation': 'sdpa',
                   'low_cpu_mem_usage': True,
                   'use_exact_model_name': True,
                   'fix_tokenizer': False,
                   'offload_embedding': False},
 'tokenizer': {'padding_side': 'right'},
 'kbit_training': {'use_gradient_checkpointing': 'unsloth'},
 'trainer': {'per_device_eval_batch_size': 1,
             'lr_scheduler_type': 'cosine',
             'warmup_steps': 10,
             'logging_steps': 10,
             'save_strategy': 'epoch',
             'eval_strategy': 'epoch',
             'save_total_limit': 2,
             'optim': 'paged_adamw_8bit',
             'dataloader_num_workers': 0,
             'dataloader_pin_memory': True,
             'report_to': 'none',
             'remove_unused_columns': False,
             'prediction_loss_only': True,
             'label_names': ['labels']},
 'collator': {'pad_to_multiple_of': 8, 'return_tensors': 'pt', 'padding': True}}

EVAL_RUNTIME_TEMPLATE = {'prefix_path': 'experiments/09/data',
 'model_name': 'Qwen/Qwen3-8B',
 'model_revision': 'b968826d9c46dd6066d109eabc6255188de91218',
 'adapter_path': 'models',
 'test_file': 'test.jsonl',
 'split_meta_file': 'split_meta.json',
 'run_config_file': 'result/run_config.json',
 'report_file': 'result/evaluation.json',
 'device': 'cuda'}

EVAL_HYPER_PARAMS = {'chat_template': {'enable_thinking': False},
 'quantization': {'load_in_4bit': True,
                  'bnb_4bit_quant_type': 'nf4',
                  'bnb_4bit_use_double_quant': True},
 'precision': {'compute_dtype': 'auto', 'frozen_embedding_dtype': 'auto'},
 'model_loading': {'device_map': {'': 0}, 'attn_implementation': 'sdpa', 'low_cpu_mem_usage': True},
 'tokenizer': {'padding_side': 'left'},
 'evaluation': {'batch_size': 1, 'max_new_tokens': 512, 'do_sample': False}}


## 2. 라이브러리와 프로젝트 경로

Unsloth를 먼저 가져옵니다. 설치는 루트의 의존성을 사용하며 노트북에서 패키지를 자동 갱신하지 않습니다.


In [2]:
import hashlib
import json
import os
import sys
import tempfile
import time
import platform
import random
from copy import deepcopy
from pathlib import Path

# 모델 저장 위치와 분리된 시스템 임시 경로에 컴파일 캐시를 둡니다.
os.environ.setdefault(
    "UNSLOTH_COMPILE_LOCATION",
    str(Path(tempfile.gettempdir()) / "siteboard_unsloth_compiled_cache"),
)
if "unsloth" not in sys.modules and any(
    name in sys.modules for name in ("transformers", "peft", "trl")
):
    raise RuntimeError("커널을 재시작한 뒤 이 노트북을 위에서부터 실행하세요. Unsloth를 먼저 가져와야 합니다.")

from unsloth import FastLanguageModel
import torch
from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
)

import hashlib
import json
import math
import platform
import re
import sys
import time
from collections import Counter
from copy import deepcopy
from pathlib import Path

import torch
from peft import PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

import gc
import statistics
from IPython.display import Markdown, display


def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / EXPERIMENT_PARAMS["experiment_dir"] / "train_desktop_unsloth.ipynb").is_file() and (candidate / "pyproject.toml").is_file():
            return candidate.resolve()
    raise FileNotFoundError("프로젝트 루트를 찾을 수 없습니다.")


PROJECT_ROOT = find_project_root()
# 파일 경로와 실행 기록을 프로젝트 루트 기준 상대 경로로 통일합니다.
os.chdir(PROJECT_ROOT)


def project_path(relative):
    path = Path(relative)
    target = (PROJECT_ROOT / path).resolve()
    if path.is_absolute() or not target.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"프로젝트 루트 기준 상대 경로를 사용하세요: {relative}")
    return target


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()


[fla.utils._device|WARNING]Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1028 23:58:58.620000 20144 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!


## 3. 학습 데이터와 같은 현장 랜덤 결합

기존 토큰화, 응답과 EOS 손실 마스킹 및 길이 제한을 유지합니다. 파트너는 자기 자신을 포함한 같은 현장 풀에서 복원 추출합니다.


In [3]:
def train_runtime_path(field):
    return project_path(Path(TRAIN_RUNTIME_PARAMS["prefix_path"]) / TRAIN_RUNTIME_PARAMS[field]).relative_to(PROJECT_ROOT).as_posix()


def train_write_json(path, value, mode="x"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open(mode, encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2)
        stream.write("\n")


def train_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def train_resolve_dtype(name):
    if name == "auto":
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return getattr(torch, name)


In [4]:
def train_read_jsonl(path):
    with Path(path).open(encoding="utf-8") as stream:
        records = [json.loads(line) for line in stream if line.strip()]
    if not records:
        raise ValueError(f"입력 JSONL이 비어 있습니다: {path}")
    return records


def train_prepare_training_data():
    train_records = train_read_jsonl(train_runtime_path("train_file"))
    valid_records = train_read_jsonl(train_runtime_path("test_file"))
    return train_records, valid_records, {
        "train_count": len(train_records), "test_count": len(valid_records),
    }


In [5]:
class QwenChatDataset:
    """같은 현장의 레코드를 랜덤 결합하고 JSON 응답 구간만 학습합니다."""

    def __init__(self, records, tokenizer, max_seq_length, is_train,
                 aug_prob, aug_max_concat, delimiters):
        self.records = records
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length
        self.is_train = is_train
        self.aug_prob = aug_prob
        self.aug_max_concat = aug_max_concat
        self.delimiters = delimiters
        self.site_to_indices = {}
        self.components = []
        for index, record in enumerate(records):
            self.site_to_indices.setdefault(record["metadata"]["site"], []).append(index)
            messages = record["messages"]
            rules, text = messages[1]["content"].split("[보드판 텍스트]\n", 1)
            self.components.append((messages[0]["content"], rules, text,
                                    json.loads(messages[2]["content"])))

    def __len__(self):
        return len(self.records)

    def encode_messages(self, messages):
        # 비추론 프롬프트와 JSON 응답을 분리해 경계 토큰까지 정확히 마스킹합니다.
        prompt = self.tokenizer.apply_chat_template(
            messages[:2], tokenize=False, add_generation_prompt=True,
            **TRAIN_HYPER_PARAMS["chat_template"],
        )
        prompt_ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        response_ids = self.tokenizer.encode(messages[2]["content"], add_special_tokens=False)
        response_ids.append(self.tokenizer.eos_token_id)
        input_ids = prompt_ids + response_ids
        if len(input_ids) > self.max_seq_length:
            raise ValueError(f"학습 입력과 정답 {len(input_ids)}토큰이 상한 {self.max_seq_length}을 넘습니다. 잘라서 학습하지 않습니다.")
        return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids),
                "labels": [-100] * len(prompt_ids) + response_ids}

    def __getitem__(self, index):
        messages = self.records[index]["messages"]
        if self.is_train and random.random() < self.aug_prob:
            system, rules, text, answers = self.components[index]
            texts, combined = [text], list(answers)
            pool = self.site_to_indices[self.records[index]["metadata"]["site"]]
            for _ in range(random.randint(2, self.aug_max_concat) - 1):
                # 06과 같이 자기 자신도 포함한 동일 현장 풀에서 복원 추출합니다.
                partner = random.choice(pool)
                _, _, partner_text, partner_answers = self.components[partner]
                texts.append(partner_text)
                combined.extend(partner_answers)
            messages = [
                {"role": "system", "content": system},
                {"role": "user", "content": rules + "[보드판 텍스트]\n" + random.choice(self.delimiters).join(texts)},
                {"role": "assistant", "content": json.dumps(combined, ensure_ascii=False, indent=2)},
            ]
        return self.encode_messages(messages)


def train_build_dataset(records, tokenizer, is_train):
    settings = TRAIN_HYPER_PARAMS["augmentation"]
    return QwenChatDataset(
        records, tokenizer, TRAIN_HYPER_PARAMS["training"]["max_seq_length"], is_train,
        settings["probability"], settings["max_concat"], settings["delimiters"],
    )


## 4. Unsloth 모델, LoRA와 Trainer


In [6]:
def train_load_tokenizer():
    # 기존 마스킹과 평가에 사용하는 공식 템플릿을 그대로 읽습니다.
    tokenizer = AutoTokenizer.from_pretrained(
        TRAIN_RUNTIME_PARAMS["model_name"], revision=TRAIN_RUNTIME_PARAMS["model_revision"],
        **TRAIN_HYPER_PARAMS["tokenizer"],
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def train_load_qlora_model():
    dtype = train_resolve_dtype(TRAIN_HYPER_PARAMS["precision"]["compute_dtype"])
    # 반환된 토크나이저 대신 위에서 읽은 공식 토크나이저로 학습하고 저장합니다.
    model, _ = FastLanguageModel.from_pretrained(
        model_name=TRAIN_RUNTIME_PARAMS["model_name"], revision=TRAIN_RUNTIME_PARAMS["model_revision"],
        max_seq_length=TRAIN_HYPER_PARAMS["training"]["max_seq_length"], dtype=dtype,
        load_in_4bit=TRAIN_HYPER_PARAMS["quantization"]["load_in_4bit"],
        quantization_config=BitsAndBytesConfig(
            bnb_4bit_compute_dtype=dtype, **TRAIN_HYPER_PARAMS["quantization"],
        ),
        use_gradient_checkpointing=TRAIN_HYPER_PARAMS["kbit_training"]["use_gradient_checkpointing"],
        random_state=TRAIN_HYPER_PARAMS["training"]["seed"],
        **TRAIN_HYPER_PARAMS["model_loading"],
    )
    actual_revision = getattr(model.config, "_commit_hash", None)
    if actual_revision is not None and actual_revision != TRAIN_RUNTIME_PARAMS["model_revision"]:
        raise ValueError(f"요청한 기본 모델 revision과 다릅니다: {actual_revision}")
    model.config.use_cache = False
    return model


def train_add_lora_adapter(model):
    settings = dict(TRAIN_HYPER_PARAMS["lora"])
    if settings.pop("task_type") != "CAUSAL_LM":
        raise ValueError("이 노트북은 CAUSAL_LM LoRA 학습만 지원합니다.")
    # Unsloth API가 k비트 준비와 LoRA 패치를 함께 수행합니다.
    model = FastLanguageModel.get_peft_model(
        model,
        random_state=TRAIN_HYPER_PARAMS["training"]["seed"],
        **settings, **TRAIN_HYPER_PARAMS["kbit_training"],
    )
    dtype = train_resolve_dtype(TRAIN_HYPER_PARAMS["precision"]["frozen_embedding_dtype"])
    for module in (model.get_input_embeddings(), model.get_output_embeddings()):
        if module is not None and not module.weight.requires_grad:
            module.to(dtype=dtype)
    for config in model.peft_config.values():
        config.base_model_name_or_path = TRAIN_RUNTIME_PARAMS["model_name"]
        config.revision = TRAIN_RUNTIME_PARAMS["model_revision"]
    model.config.use_cache = False
    model.print_trainable_parameters()
    return model


def train_build_trainer(model, tokenizer, train_records, valid_records):
    settings = TRAIN_HYPER_PARAMS["training"]
    train_dataset = train_build_dataset(train_records, tokenizer, is_train=True)
    valid_dataset = train_build_dataset(valid_records, tokenizer, is_train=False)
    dtype = train_resolve_dtype(TRAIN_HYPER_PARAMS["precision"]["compute_dtype"])
    args = TrainingArguments(
        output_dir=train_runtime_path("output_dir"), num_train_epochs=settings["epochs"],
        per_device_train_batch_size=settings["batch_size"],
        gradient_accumulation_steps=settings["grad_accum"],
        learning_rate=settings["learning_rate"], seed=settings["seed"],
        bf16=dtype == torch.bfloat16, fp16=dtype == torch.float16,
        **TRAIN_HYPER_PARAMS["trainer"],
    )
    # Dataset을 직접 전달해 매 접근 시점의 랜덤 결합과 labels를 유지합니다.
    trainer = Trainer(
        model=model, args=args, processing_class=tokenizer,
        train_dataset=train_dataset, eval_dataset=valid_dataset,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, **TRAIN_HYPER_PARAMS["collator"]),
    )
    return trainer


In [7]:
def train_collect_execution_metadata(model=None):
    from importlib.metadata import PackageNotFoundError, version

    packages = {}
    for name in ("torch", "transformers", "peft", "accelerate", "bitsandbytes",
                 "unsloth", "unsloth_zoo", "trl", "triton", "triton-windows",
                 "torchvision", "xformers"):
        try:
            packages[name] = version(name)
        except PackageNotFoundError:
            packages[name] = None
    return {
        "python": platform.python_version(), "python_executable": sys.executable,
        "platform": platform.platform(), "packages": packages,
        "training_backend": "unsloth", "trainer": "transformers.Trainer",
        "gradient_checkpointing": TRAIN_HYPER_PARAMS["kbit_training"]["use_gradient_checkpointing"],
        "unsloth_compile_location": os.environ.get("UNSLOTH_COMPILE_LOCATION"),
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "cuda": torch.version.cuda,
        "base_model": TRAIN_RUNTIME_PARAMS["model_name"],
        "base_model_revision": getattr(getattr(model, "config", None), "_commit_hash", None)
            or TRAIN_RUNTIME_PARAMS["model_revision"],
        "compute_dtype": str(train_resolve_dtype(TRAIN_HYPER_PARAMS["precision"]["compute_dtype"])),
        "input_embedding_dtype": str(model.get_input_embeddings().weight.dtype) if model is not None else None,
        "output_embedding_dtype": str(model.get_output_embeddings().weight.dtype) if model is not None else None,
        "gpu_total_bytes": torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
        "peak_allocated_bytes": torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None,
        "peak_reserved_bytes": torch.cuda.max_memory_reserved() if torch.cuda.is_available() else None,
    }


def train_check_training_environment():
    if TRAIN_RUNTIME_PARAMS["device"] != "cuda" or not torch.cuda.is_available():
        raise RuntimeError("이 노트북은 CUDA GPU 런타임이 필요합니다.")
    dtype = train_resolve_dtype(TRAIN_HYPER_PARAMS["precision"]["compute_dtype"])
    if dtype not in (torch.bfloat16, torch.float16):
        raise ValueError("compute_dtype은 auto, bfloat16 또는 float16을 사용하세요.")
    if dtype == torch.bfloat16 and not torch.cuda.is_bf16_supported():
        raise ValueError("이 GPU는 BF16을 지원하지 않습니다. compute_dtype을 float16으로 지정하세요.")
    for key in ("train_file", "test_file"):
        if not Path(train_runtime_path(key)).is_file():
            raise FileNotFoundError(f"입력 JSONL이 없습니다. prefix_path를 확인하세요: {runtime_path(key)}")


def train_check_output_paths():
    for key in ("run_config_file", "training_history_file", "report_file"):
        if Path(train_runtime_path(key)).exists():
            raise FileExistsError(f"기존 산출물이 있습니다: {runtime_path(key)}")
    adapter = Path(train_runtime_path("adapter_path"))
    if adapter.exists() and any(p.name != "README.md" for p in adapter.iterdir()):
        raise FileExistsError(f"기존 모델 또는 체크포인트가 있습니다: {adapter}")
    output = Path(train_runtime_path("output_dir"))
    if output.exists() and any(output.iterdir()):
        raise FileExistsError(f"기존 체크포인트가 있습니다: {output}")


## 5. 평가

저장된 어댑터를 다시 읽습니다. strict_schema_multiset_v1 채점과 값 토큰 신뢰도 계산은 기존 구현을 유지하며 함수와 설정의 이름은 학습과 분리합니다.


In [8]:
def eval_runtime_path(field):
    return project_path(Path(EVAL_RUNTIME_PARAMS["prefix_path"]) / EVAL_RUNTIME_PARAMS[field]).relative_to(PROJECT_ROOT).as_posix()


def eval_write_json(path, value, mode="x"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open(mode, encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2)
        stream.write("\n")


def eval_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def eval_resolve_dtype(name):
    if name == "auto":
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return getattr(torch, name)


In [9]:
def eval_read_jsonl(path):
    with Path(path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


In [10]:
def eval_unique_object(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"중복 JSON 키: {key}")
        result[key] = value
    return result


def eval_parse_prediction(text):
    """필드와 값 타입을 검사하고, 유효하지 않은 출력은 None으로 반환합니다."""
    try:
        data = json.loads(text, object_pairs_hook=eval_unique_object)
        if not isinstance(data, list):
            return None
        for item in data:
            if not isinstance(item, dict) or set(item) != {"workType", "spec", "quantity"}:
                return None
            if not isinstance(item["workType"], str) or not item["workType"].strip():
                return None
            if not isinstance(item["spec"], str):
                return None
            item["workType"].encode("utf-8")
            item["spec"].encode("utf-8")
            if type(item["quantity"]) not in (int, float) or not math.isfinite(item["quantity"]):
                return None
        return data
    except (ValueError, TypeError, OverflowError, RecursionError):
        return None


def eval_parse_and_normalize(text):
    data = eval_parse_prediction(text)
    if data is None:
        return None
    return [(item["workType"].strip(), item["spec"].strip(), round(item["quantity"], 4)) for item in data]


def eval_build_token_spans(tokenizer, token_ids, probabilities, pred_text):
    """Qwen 바이트 토큰을 복원해 한글 분할 토큰에도 같은 값 구간을 연결합니다."""
    visible = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
    missing = [value for value in range(256) if value not in visible]
    byte_decoder = dict(zip(map(chr, visible + list(range(256, 256 + len(missing)))), visible + missing))
    spans = []
    restored = bytearray()
    try:
        for token, probability in zip(tokenizer.convert_ids_to_tokens(token_ids), probabilities):
            start = len(restored)
            restored.extend(byte_decoder[char] for char in token)
            spans.append((start, len(restored), probability))
    except KeyError:
        return None
    if len(spans) != len(token_ids) or bytes(restored) != pred_text.encode("utf-8"):
        return None
    return spans


def eval_extract_values_with_confidence(pred_text, token_spans):
    """값과 겹치는 생성 토큰을 집계하며 대응이 미확인인 값은 None으로 둡니다."""
    data = eval_parse_prediction(pred_text)
    if data is None:
        return [], None, None
    predictions = [dict(item) for item in data]
    for item in predictions:
        for field in ("workType", "spec", "quantity"):
            item[f"{field}_confidence"] = None
    if token_spans is None:
        return predictions, None, None

    # JSON 문자열 전체를 먼저 매칭해 값 내부의 키처럼 보이는 문자열을 제외합니다.
    string = r'"(?:[^"\\\x00-\x1f]|\\(?:["\\/bfnrt]|u[0-9a-fA-F]{4}))*"'
    number = r'-?(?:0|[1-9][0-9]*)(?:\.[0-9]+)?(?:[eE][+-]?[0-9]+)?'
    fields = list(re.finditer(rf'({string})\s*:\s*({string}|{number})|{string}', pred_text))
    fields = [match for match in fields if match.group(1) is not None]
    if len(fields) != 3 * len(data):
        return predictions, None, None
    byte_offsets = [0]
    for char in pred_text:
        byte_offsets.append(byte_offsets[-1] + len(char.encode("utf-8")))
    value_indices = set()
    for index, match in enumerate(fields):
        field = json.loads(match.group(1))
        start, end = match.span(2)
        if pred_text[start] == '"':
            start, end = start + 1, end - 1
        selected = {i for i, (left, right, _) in enumerate(token_spans)
                    if max(byte_offsets[start], left) < min(byte_offsets[end], right)}
        if selected:
            probabilities = [token_spans[i][2] for i in selected]
            predictions[index // 3][f"{field}_confidence"] = sum(probabilities) / len(probabilities)
            value_indices.update(selected)
    probabilities = [token_spans[i][2] for i in value_indices]
    return (predictions, sum(probabilities) / len(probabilities), min(probabilities)) if probabilities else (predictions, None, None)


def eval_summarize_cases(cases):
    values = [case["value_min_confidence"] for case in cases if case["value_min_confidence"] is not None]
    averages = [case["value_avg_confidence"] for case in cases if case["value_avg_confidence"] is not None]
    correct = sum(case["is_exact_match"] for case in cases)
    return {
        "total": len(cases),
        "correct": correct,
        "accuracy": 100.0 * correct / len(cases) if cases else None,
        "value_min_confidence": min(values) if values else None,
        "value_avg_confidence": sum(averages) / len(averages) if averages else None,
        "confidence_count": len(values),
        "confidence_missing_count": len(cases) - len(values),
    }


In [11]:
def eval_load_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(
        eval_runtime_path("adapter_path"), **EVAL_HYPER_PARAMS["tokenizer"],
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def eval_load_model():
    dtype = eval_resolve_dtype(EVAL_HYPER_PARAMS["precision"]["compute_dtype"])
    embedding_dtype = eval_resolve_dtype(EVAL_HYPER_PARAMS["precision"]["frozen_embedding_dtype"])
    model = AutoModelForCausalLM.from_pretrained(
        EVAL_RUNTIME_PARAMS["model_name"], revision=EVAL_RUNTIME_PARAMS["model_revision"],
        dtype=dtype,
        quantization_config=BitsAndBytesConfig(
            bnb_4bit_compute_dtype=dtype, **EVAL_HYPER_PARAMS["quantization"],
        ),
        **EVAL_HYPER_PARAMS["model_loading"],
    )
    for module in model.modules():
        if "norm" in type(module).__name__.lower():
            module.to(dtype=torch.float32)
    for module in (model.get_input_embeddings(), model.get_output_embeddings()):
        module.to(dtype=embedding_dtype)
    model = PeftModel.from_pretrained(model, eval_runtime_path("adapter_path"), is_trainable=False)
    model.config.use_cache = True
    model.eval()
    return model


In [12]:
def eval_prepare_evaluation(tokenizer):
    samples = eval_read_jsonl(eval_runtime_path("test_file"))
    prompts = [tokenizer.apply_chat_template(
        sample["messages"][:2], tokenize=False, add_generation_prompt=True,
        **EVAL_HYPER_PARAMS["chat_template"],
    ) for sample in samples]
    paths = [eval_runtime_path(key) for key in ("test_file", "split_meta_file", "run_config_file")]
    adapter = Path(eval_runtime_path("adapter_path"))
    paths.extend(str(adapter / name) for name in
                 ("adapter_config.json", "adapter_model.safetensors", "tokenizer.json", "tokenizer_config.json"))
    return samples, prompts, {path: eval_file_sha256(path) for path in paths}


In [13]:
def eval_collect_execution_metadata(model=None):
    from importlib.metadata import PackageNotFoundError, version

    packages = {}
    for name in ("torch", "transformers", "peft", "accelerate", "bitsandbytes"):
        try:
            packages[name] = version(name)
        except PackageNotFoundError:
            packages[name] = None
    return {
        "python": platform.python_version(), "python_executable": sys.executable,
        "platform": platform.platform(), "packages": packages,
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "cuda": torch.version.cuda,
        "base_model": EVAL_RUNTIME_PARAMS["model_name"],
        "base_model_revision": getattr(getattr(model, "config", None), "_commit_hash", None)
            or EVAL_RUNTIME_PARAMS["model_revision"],
        "compute_dtype": str(eval_resolve_dtype(EVAL_HYPER_PARAMS["precision"]["compute_dtype"])),
        "input_embedding_dtype": str(model.get_input_embeddings().weight.dtype) if model else None,
        "output_embedding_dtype": str(model.get_output_embeddings().weight.dtype) if model else None,
        "gpu_total_bytes": torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
        "peak_allocated_bytes": torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None,
        "peak_reserved_bytes": torch.cuda.max_memory_reserved() if torch.cuda.is_available() else None,
    }


In [14]:
def eval_evaluate():
    report_path = Path(eval_runtime_path("report_file"))
    if report_path.exists():
        raise FileExistsError(f"기존 평가 결과가 있습니다: {report_path}")
    settings = EVAL_HYPER_PARAMS["evaluation"]
    tokenizer = eval_load_tokenizer()
    samples, prompts, file_hashes = eval_prepare_evaluation(tokenizer)
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    model = eval_load_model()
    eos_ids = {tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")}
    eos_ids.discard(None)
    case_results = []
    print(f"평가 {len(samples)}건 / GPU: {torch.cuda.get_device_name(0)}")

    for start in tqdm(range(0, len(samples), settings["batch_size"]), desc="Evaluation"):
        batch = samples[start:start + settings["batch_size"]]
        inputs = tokenizer(prompts[start:start + len(batch)], return_tensors="pt", padding=True,
                           truncation=False, add_special_tokens=False).to(model.device)
        with torch.inference_mode(), torch.autocast(
                device_type="cuda", dtype=eval_resolve_dtype(EVAL_HYPER_PARAMS["precision"]["compute_dtype"])):
            outputs = model.generate(
                **inputs,
                max_new_tokens=settings["max_new_tokens"],
                do_sample=settings["do_sample"],
                temperature=None, top_p=None, top_k=None,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=sorted(eos_ids),
                return_dict_in_generate=True,
                output_scores=True,
            )
        generated = outputs.sequences[:, inputs["input_ids"].shape[1]:]
        for offset, sample in enumerate(batch):
            token_ids = generated[offset].tolist()
            end = next((i for i, token_id in enumerate(token_ids)
                        if token_id in eos_ids or token_id == tokenizer.pad_token_id), len(token_ids))
            terminated = end < len(token_ids) and token_ids[end] in eos_ids
            token_ids = token_ids[:end]
            pred_text = tokenizer.decode(token_ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)
            probabilities = [torch.softmax(outputs.scores[i][offset].float(), dim=-1)[token_id].item()
                             for i, token_id in enumerate(token_ids)]
            token_spans = eval_build_token_spans(tokenizer, token_ids, probabilities, pred_text)
            prediction, avg_confidence, min_confidence = eval_extract_values_with_confidence(pred_text, token_spans)
            ground_truth = sample["messages"][2]["content"]
            norm_pred = eval_parse_and_normalize(pred_text)
            is_match = norm_pred is not None and Counter(norm_pred) == Counter(eval_parse_and_normalize(ground_truth))
            metadata = sample["metadata"]
            case_results.append({
                "case_id": start + offset + 1,
                **{key: metadata[key] for key in ("site", "image_id", "source_index", "test_type")},
                "input_user": sample["messages"][1]["content"],
                "ground_truth_raw": ground_truth,
                "prediction_raw": pred_text,
                "prediction": prediction,
                "is_valid_json_schema": norm_pred is not None,
                "is_exact_match": is_match,
                "output_limit_reached": not terminated and end >= settings["max_new_tokens"],
                "value_avg_confidence": avg_confidence,
                "value_min_confidence": min_confidence,
            })
        # 다음 배치를 생성하기 전에 어휘 점수를 해제합니다.
        del outputs, inputs, generated

    overall = eval_summarize_cases(case_results)
    by_type = {name: eval_summarize_cases([case for case in case_results if case["test_type"] == name])
               for name in sorted({case["test_type"] for case in case_results})}
    report = {
        "experiment": deepcopy(ACTIVE_RUN_IDENTITY),
        "model": EVAL_RUNTIME_PARAMS["model_name"],
        "adapter": eval_runtime_path("adapter_path"),
        "device": torch.cuda.get_device_name(0),
        "dtype": str(model.dtype),
        "quantized": True, "adapter_merged": False,
        "duration_seconds": time.perf_counter() - started,
        "test_file": eval_runtime_path("test_file"),
        "runtime_params": deepcopy(EVAL_RUNTIME_PARAMS),
        "hyper_params": deepcopy(EVAL_HYPER_PARAMS),
        "batch_size": settings["batch_size"],
        "scoring_version": "strict_schema_multiset_v1",
        "confidence_definition": "Generated tokens overlapping workType/spec/quantity value bytes; unique tokens per case. Group min includes correct and incorrect cases with known confidence; null cases are excluded and counted. Avg is the mean of known case averages.",
        "file_hashes": file_hashes,
        "execution": eval_collect_execution_metadata(model),
        **overall,
        "correct_value_avg_confidence": eval_summarize_cases([case for case in case_results if case["is_exact_match"]])["value_avg_confidence"],
        "incorrect_value_avg_confidence": eval_summarize_cases([case for case in case_results if not case["is_exact_match"]])["value_avg_confidence"],
        "by_test_type": by_type,
        "cases": case_results,
    }
    eval_write_json(report_path, report)
    print(f"완전 일치율: {overall['accuracy']:.2f}% ({overall['correct']}/{overall['total']})")
    print(f"평가 결과 저장: {report_path}")
    return report


## 6. 실행 조건과 산출물 정합성

완료된 조건만 검증 후 건너뜁니다. 학습 완료 후 평가만 재실행할 수 있습니다. 미완료 학습이나 변경된 입력, 설정, 모델이 발견되면 중단합니다.


In [15]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def notebook_code_hash():
    document = read_json(project_path(Path(EXPERIMENT_PARAMS["experiment_dir"]) / "train_desktop_unsloth.ipynb"))
    sources = ["".join(cell["source"]) for cell in document["cells"] if cell["cell_type"] == "code"]
    return hashlib.sha256(json.dumps(sources, ensure_ascii=False).encode("utf-8")).hexdigest()


def input_hashes():
    base = Path(TRAIN_RUNTIME_TEMPLATE["prefix_path"])
    return {(base / name).as_posix(): train_file_sha256(project_path(base / name)) for name in
            ("work_type.json", "cluster.json", "split_meta.json", "train.jsonl", "test.jsonl")}


def artifact_hashes(adapter):
    return {name: train_file_sha256(Path(adapter) / name) for name in
            ("adapter_config.json", "adapter_model.safetensors", "tokenizer.json", "tokenizer_config.json")}


def environment_signature():
    metadata = train_collect_execution_metadata()
    return {key: metadata[key] for key in ("python", "platform", "packages", "device", "cuda",
            "compute_dtype", "training_backend", "base_model", "base_model_revision")}


def configure_condition(seed, mode):
    global TRAIN_RUNTIME_PARAMS, TRAIN_HYPER_PARAMS, EVAL_RUNTIME_PARAMS, ACTIVE_RUN_IDENTITY
    run_id = f"seed_{seed}_{mode}"
    TRAIN_RUNTIME_PARAMS = deepcopy(TRAIN_RUNTIME_TEMPLATE)
    TRAIN_HYPER_PARAMS = deepcopy(TRAIN_HYPER_TEMPLATE)
    TRAIN_HYPER_PARAMS["training"]["seed"] = seed
    TRAIN_HYPER_PARAMS["augmentation"]["probability"] = EXPERIMENT_PARAMS["augmentation_modes"][mode]
    TRAIN_RUNTIME_PARAMS.update(
        adapter_path=f"models/{run_id}", output_dir=f"models/{run_id}/checkpoints",
        run_config_file=f"result/{run_id}.run_config.json",
        training_history_file=f"result/{run_id}.training_history.json",
        report_file=f"result/{run_id}.evaluation.json",
    )
    EVAL_RUNTIME_PARAMS = deepcopy(EVAL_RUNTIME_TEMPLATE)
    EVAL_RUNTIME_PARAMS.update({key: TRAIN_RUNTIME_PARAMS[key] for key in
        ("prefix_path", "model_name", "model_revision", "adapter_path", "test_file",
         "run_config_file", "report_file", "device")})
    ACTIVE_RUN_IDENTITY = {"run_id": run_id, "seed": seed, "augmentation": mode,
                           "implementation_sha256": notebook_code_hash()}
    return run_id


def check_experiment_inputs():
    seeds = EXPERIMENT_PARAMS["seeds"]
    if len(seeds) != 3 or len(set(seeds)) != 3 or any(type(s) is not int or s < 0 or s == 42 for s in seeds):
        raise ValueError("42를 제외한 서로 다른 비음수 정수 학습 시드 3개가 필요합니다.")
    modes = EXPERIMENT_PARAMS["augmentation_modes"]
    if list(modes) != ["on", "off"] or modes != {"on": 0.5, "off": 0.0}:
        raise ValueError("증강 조건은 on=0.5, off=0.0 순서여야 합니다.")
    expected_base = (Path(EXPERIMENT_PARAMS["experiment_dir"]) / "data").as_posix()
    if TRAIN_RUNTIME_TEMPLATE["prefix_path"] != expected_base or EVAL_RUNTIME_TEMPLATE["prefix_path"] != expected_base:
        raise ValueError("학습과 평가 모두 09 내부의 같은 데이터 경로를 사용하세요.")
    for key in ("model_name", "model_revision"):
        if TRAIN_RUNTIME_TEMPLATE[key] != EVAL_RUNTIME_TEMPLATE[key]:
            raise ValueError(f"학습과 평가의 {key} 설정이 다릅니다.")
    meta = read_json(project_path(Path(expected_base) / "split_meta.json"))
    if meta["seed"] != EXPERIMENT_PARAMS["split_seed"] or meta["seed"] != 42:
        raise ValueError("분할 시드는 42여야 합니다.")
    if meta["train"]["count"] != EXPERIMENT_PARAMS["expected_train_count"] or meta["test"]["total_count"] != EXPERIMENT_PARAMS["expected_test_count"]:
        raise ValueError("분할 건수가 다릅니다.")
    return input_hashes()


def inspect_condition(expected_inputs, expected_environment):
    config_path = Path(train_runtime_path("run_config_file"))
    history_path = Path(train_runtime_path("training_history_file"))
    report_path = Path(eval_runtime_path("report_file"))
    if not config_path.exists():
        train_check_output_paths()
        return "train"
    config = read_json(config_path)
    if config.get("status") != "training_completed":
        raise RuntimeError(f"미완료 학습을 자동 재개하거나 덮어쓰지 않습니다: {config_path}")
    expected = {"experiment": ACTIVE_RUN_IDENTITY, "runtime_params": TRAIN_RUNTIME_PARAMS,
                "hyper_params": TRAIN_HYPER_PARAMS, "input_file_hashes": expected_inputs,
                "environment_signature": expected_environment}
    for field, value in expected.items():
        if config.get(field) != value:
            raise ValueError(f"완료된 학습의 {field} 불일치: {config_path}")
    if config.get("artifact_hashes") != artifact_hashes(train_runtime_path("adapter_path")):
        raise ValueError(f"저장 모델 해시 불일치: {config_path}")
    if config.get("training_history_sha256") != train_file_sha256(history_path):
        raise ValueError(f"학습 이력 해시 불일치: {history_path}")
    settings = TRAIN_HYPER_PARAMS["training"]
    steps = math.ceil(EXPERIMENT_PARAMS["expected_train_count"] / (settings["batch_size"] * settings["grad_accum"])) * settings["epochs"]
    if config.get("global_step") != steps or config.get("epoch") != settings["epochs"]:
        raise ValueError("학습 스텝 또는 에포크가 설정과 다릅니다.")
    if not report_path.exists():
        return "evaluate"
    report = read_json(report_path)
    for field, value in {"experiment": ACTIVE_RUN_IDENTITY, "runtime_params": EVAL_RUNTIME_PARAMS,
                         "hyper_params": EVAL_HYPER_PARAMS, "scoring_version": "strict_schema_multiset_v1"}.items():
        if report.get(field) != value:
            raise ValueError(f"평가 {field} 불일치: {report_path}")
    paths = [eval_runtime_path(key) for key in ("test_file", "split_meta_file", "run_config_file")]
    paths += [str(Path(eval_runtime_path("adapter_path")) / name) for name in config["artifact_hashes"]]
    if report.get("file_hashes") != {path: eval_file_sha256(path) for path in paths}:
        raise ValueError(f"평가 입력 또는 모델 해시 불일치: {report_path}")
    validate_report(report)
    return "complete"


## 7. 조건별 학습과 저장

기존 학습 본체에 실행 상태, 완료 스텝, 모델과 이력 해시를 추가합니다. 학습 실패 시 오류를 보존하고 전체 반복을 멈춥니다.


In [16]:
def run_condition_training(expected_inputs, expected_environment):
    train_check_output_paths()
    train_check_training_environment()
    set_seed(TRAIN_HYPER_PARAMS["training"]["seed"])
    train_records, valid_records, prepared_data = train_prepare_training_data()
    if prepared_data != {"train_count": EXPERIMENT_PARAMS["expected_train_count"], "test_count": EXPERIMENT_PARAMS["expected_test_count"]}:
        raise ValueError("학습 또는 평가 레코드 수가 다릅니다.")
    started = time.perf_counter()
    config = {
        "experiment": deepcopy(ACTIVE_RUN_IDENTITY),
        "runtime_params": deepcopy(TRAIN_RUNTIME_PARAMS), "hyper_params": deepcopy(TRAIN_HYPER_PARAMS),
        "prepared_data": prepared_data, "input_file_hashes": expected_inputs,
        "prepared_file_hashes": {train_runtime_path(key): train_file_sha256(train_runtime_path(key)) for key in ("train_file", "test_file")},
        "environment_signature": expected_environment, "status": "training_started",
    }
    train_write_json(train_runtime_path("run_config_file"), config)
    model = trainer = tokenizer = None
    try:
        tokenizer = train_load_tokenizer()
        torch.cuda.reset_peak_memory_stats()
        model = train_add_lora_adapter(train_load_qlora_model())
        trainer = train_build_trainer(model, tokenizer, train_records, valid_records)
        trainer.train()
        destination = train_runtime_path("adapter_path")
        model.save_pretrained(destination)
        tokenizer.save_pretrained(destination)
        train_write_json(train_runtime_path("training_history_file"), trainer.state.log_history)
        config.update(artifact_hashes=artifact_hashes(destination),
                      training_history_sha256=train_file_sha256(train_runtime_path("training_history_file")),
                      global_step=trainer.state.global_step, epoch=trainer.state.epoch,
                      execution=train_collect_execution_metadata(model), status="training_completed",
                      duration_seconds=time.perf_counter() - started)
        train_write_json(train_runtime_path("run_config_file"), config, mode="w")
    except (Exception, KeyboardInterrupt) as error:
        config.update(status="training_interrupted" if isinstance(error, KeyboardInterrupt) else "training_failed",
                      error={"type": type(error).__name__, "message": str(error)},
                      duration_seconds=time.perf_counter() - started)
        train_write_json(train_runtime_path("run_config_file"), config, mode="w")
        if trainer is not None and not Path(train_runtime_path("training_history_file")).exists():
            train_write_json(train_runtime_path("training_history_file"), trainer.state.log_history)
        raise
    finally:
        model = trainer = tokenizer = None
        train_records = valid_records = None
        cleanup_cuda()


## 8. 평가 기록 검증과 비교

식별자, 입력과 정답, 유형을 전수 대응합니다. 저장된 채점을 보존하면서 같은 기준으로 확인합니다. 값 토큰 최소 신뢰도는 정답과 오답을 포함한 최솟값이며, 누락은 null로 유지합니다.


In [17]:
def case_key(case):
    return tuple(case[key] for key in ("site", "image_id", "source_index"))


def test_index():
    samples = eval_read_jsonl(eval_runtime_path("test_file"))
    index = {case_key(sample["metadata"]): sample for sample in samples}
    if len(index) != len(samples) or len(samples) != EXPERIMENT_PARAMS["expected_test_count"]:
        raise ValueError("평가 식별자 중복 또는 건수 불일치입니다.")
    meta = read_json(eval_runtime_path("split_meta_file"))
    for sample in samples:
        m = sample["metadata"]
        if m["source_index"] not in meta["test"]["subsets"][m["test_type"]]["indices"]:
            raise ValueError("평가 유형과 분할 메타데이터가 다릅니다.")
    return index


def validate_report(report):
    expected = test_index()
    cases = report["cases"]
    actual = {case_key(case): case for case in cases}
    if len(actual) != len(cases) or actual.keys() != expected.keys():
        raise ValueError("평가 케이스 식별자가 일치하지 않습니다.")
    for key, case in actual.items():
        sample = expected[key]
        if case["input_user"] != sample["messages"][1]["content"] or case["ground_truth_raw"] != sample["messages"][2]["content"] or case["test_type"] != sample["metadata"]["test_type"]:
            raise ValueError(f"입력, 정답 또는 유형 불일치: {key}")
        pred = eval_parse_and_normalize(case["prediction_raw"])
        correct = pred is not None and Counter(pred) == Counter(eval_parse_and_normalize(case["ground_truth_raw"]))
        if correct != case["is_exact_match"] or (pred is not None) != case["is_valid_json_schema"]:
            raise ValueError(f"저장된 채점이 다릅니다: {key}")
    for field, value in eval_summarize_cases(cases).items():
        if report.get(field) != value:
            raise ValueError(f"전체 집계 불일치: {field}")
    groups = {name: eval_summarize_cases([c for c in cases if c["test_type"] == name]) for name in sorted({c["test_type"] for c in cases})}
    if groups != report.get("by_test_type"):
        raise ValueError("유형별 집계가 다릅니다.")
    return actual


def compare_group(on, off, keys):
    a = eval_summarize_cases([on[key] for key in keys])
    b = eval_summarize_cases([off[key] for key in keys])
    transitions = Counter((on[key]["is_exact_match"], off[key]["is_exact_match"]) for key in keys)
    only_on, only_off = transitions[True, False], transitions[False, True]
    if a["correct"] - b["correct"] != only_on - only_off:
        raise ValueError("정답 차이와 전환 건수 차이가 다릅니다.")
    return {"on": a, "off": b, "correct_delta": a["correct"] - b["correct"],
            "accuracy_delta_pp": a["accuracy"] - b["accuracy"] if keys else None,
            "both_correct": transitions[True, True], "on_only_correct": only_on,
            "off_only_correct": only_off, "both_incorrect": transitions[False, False]}


def build_comparison(reports):
    index = test_index()
    groups = {"overall": list(index)}
    for name in sorted({x["metadata"]["test_type"] for x in index.values()}):
        groups[name] = [key for key, x in index.items() if x["metadata"]["test_type"] == name]
    groups["without_cluster_22"] = [key for key, x in index.items() if x["metadata"]["cluster_id"] != EXPERIMENT_PARAMS["excluded_cluster"]]
    groups["one_task"] = [key for key, x in index.items() if len(json.loads(x["messages"][2]["content"])) == 1]
    groups["multiple_tasks"] = [key for key, x in index.items() if len(json.loads(x["messages"][2]["content"])) >= 2]
    for cluster in sorted({x["metadata"]["cluster_id"] for x in index.values()}):
        groups[f"cluster_{cluster}"] = [key for key, x in index.items() if x["metadata"]["cluster_id"] == cluster]
    paired, summaries = [], []
    expected_ids = {f"seed_{seed}_{mode}" for seed in EXPERIMENT_PARAMS["seeds"] for mode in EXPERIMENT_PARAMS["augmentation_modes"]}
    if set(reports) != expected_ids:
        raise ValueError("6개 조건의 평가가 모두 필요합니다.")
    for seed in EXPERIMENT_PARAMS["seeds"]:
        pair = {}
        for mode in EXPERIMENT_PARAMS["augmentation_modes"]:
            run_id = f"seed_{seed}_{mode}"
            report = reports[run_id]
            identity = report["experiment"]
            if identity != {"run_id": run_id, "seed": seed, "augmentation": mode, "implementation_sha256": notebook_code_hash()}:
                raise ValueError(f"실행 식별자가 다릅니다: {run_id}")
            pair[mode] = validate_report(report)
            summaries.append({"run_id": run_id, "seed": seed, "augmentation": mode,
                              "overall": eval_summarize_cases(report["cases"]), "by_test_type": report["by_test_type"]})
        paired.append({"seed": seed, "groups": {name: compare_group(pair["on"], pair["off"], keys) for name, keys in groups.items()}})
    aggregate = {}
    for name in groups:
        values = [row["groups"][name]["accuracy_delta_pp"] for row in paired]
        aggregate[name] = {"seed_count": len(values), "mean_accuracy_delta_pp": statistics.mean(values),
                           "sample_std_accuracy_delta_pp": statistics.stdev(values),
                           "min_accuracy_delta_pp": min(values), "max_accuracy_delta_pp": max(values),
                           "on_wins": sum(v > 0 for v in values), "ties": sum(v == 0 for v in values),
                           "off_wins": sum(v < 0 for v in values)}
    base = Path(TRAIN_RUNTIME_TEMPLATE["prefix_path"])
    return {"experiment": "09", "seeds": list(EXPERIMENT_PARAMS["seeds"]),
            "implementation_sha256": notebook_code_hash(), "input_file_hashes": input_hashes(),
            "evaluation_sha256": {run_id: eval_file_sha256(project_path(base / "result" / f"{run_id}.evaluation.json")) for run_id in reports},
            "confidence_definition": "min of known case value_min_confidence including correct and incorrect cases; JSON syntax excluded; missing remains null",
            "interpretation": "동일 스케줄의 증강 효과와 시드별 반복성. 토큰 노출량 대조 없음. 시드 42 제외. 통계적 유의성 자동 판정 없음.",
            "runs": summaries, "paired": paired, "across_seeds": aggregate}


In [18]:
def display_comparison(comparison):
    def table(headers, rows):
        def fmt(value):
            if value is None:
                return "미확인"
            return f"{value:.6f}" if isinstance(value, float) else str(value)
        lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
        lines += ["| " + " | ".join(fmt(v) for v in row) + " |" for row in rows]
        display(Markdown("\n".join(lines)))
    display(Markdown("### 실행별 전체 및 유형별 결과"))
    rows = []
    for run in comparison["runs"]:
        for group, x in {"overall": run["overall"], **run["by_test_type"]}.items():
            rows.append([run["run_id"], group, x["total"], x["correct"], x["accuracy"], x["value_min_confidence"], x["confidence_missing_count"]])
    table(["실행", "유형", "건수", "정답", "일치율 (%)", "값 토큰 최소 신뢰도", "신뢰도 미확인"], rows)
    display(Markdown("### 시드별 on − off 및 정답 전환"))
    rows = [[pair["seed"], name, g["correct_delta"], g["accuracy_delta_pp"], g["on_only_correct"], g["off_only_correct"], g["both_correct"], g["both_incorrect"]]
            for pair in comparison["paired"] for name, g in pair["groups"].items() if not name.startswith("cluster_")]
    table(["시드", "범위", "정답 차이", "일치율 차이 (%p)", "on만 정답", "off만 정답", "모두 정답", "모두 오답"], rows)
    display(Markdown("### 시드 간 차이 집계"))
    rows = [[name, x["mean_accuracy_delta_pp"], x["sample_std_accuracy_delta_pp"], x["min_accuracy_delta_pp"], x["max_accuracy_delta_pp"], x["on_wins"], x["ties"], x["off_wins"]]
            for name, x in comparison["across_seeds"].items() if not name.startswith("cluster_")]
    table(["범위", "평균 (%p)", "표본 표준편차", "최소", "최대", "on 우세", "동률", "off 우세"], rows)
    display(Markdown("### 평가 클러스터별 비교"))
    rows = [[pair["seed"], name, g["on"]["total"], g["on"]["correct"], g["off"]["correct"], g["accuracy_delta_pp"]]
            for pair in comparison["paired"] for name, g in pair["groups"].items() if name.startswith("cluster_")]
    table(["시드", "클러스터", "평가 건수", "on 정답", "off 정답", "차이 (%p)"], rows)
    display(Markdown("값 토큰 최소 신뢰도는 정답과 오답을 포함한 케이스별 최솟값의 min입니다. 평균이나 보정된 정답 확률이 아닙니다. 3개 시드의 평균이 양수여도 통계적 유의성을 자동 판정하지 않습니다."))


## 9. 전체 자동 실행

아래 셀부터 실제 GPU 학습이 시작됩니다. 각 조건은 학습, 메모리 정리, 평가, 완료 검증 순으로 진행합니다. 첫 실패에서 멈추고 기존 결과는 보존합니다.


In [19]:
def run_experiment():
    expected_inputs = check_experiment_inputs()
    reports = {}
    baseline_environment = None
    for seed in EXPERIMENT_PARAMS["seeds"]:
        for mode in EXPERIMENT_PARAMS["augmentation_modes"]:
            run_id = configure_condition(seed, mode)
            stage = "사전 확인"
            try:
                train_check_training_environment()
                environment = environment_signature()
                if baseline_environment is None:
                    baseline_environment = environment
                elif environment != baseline_environment:
                    raise ValueError("실행 간 환경이 달라졌습니다.")
                if input_hashes() != expected_inputs:
                    raise ValueError("실행 중 입력 파일이 변경됐습니다.")
                state = inspect_condition(expected_inputs, environment)
                print(f"[{run_id}] {state}", flush=True)
                if state == "train":
                    stage = "학습"
                    run_condition_training(expected_inputs, environment)
                    cleanup_cuda()
                    state = inspect_condition(expected_inputs, environment)
                if state == "evaluate":
                    stage = "평가"
                    eval_evaluate()
                    cleanup_cuda()
                stage = "완료 검증"
                if input_hashes() != expected_inputs:
                    raise ValueError("학습 또는 평가 중 입력 파일이 변경됐습니다.")
                if inspect_condition(expected_inputs, environment) != "complete":
                    raise RuntimeError("학습 또는 평가가 완료되지 않았습니다.")
                reports[run_id] = read_json(eval_runtime_path("report_file"))
                print(f"[{run_id}] 완료", flush=True)
            except (Exception, KeyboardInterrupt) as error:
                print(f"[{run_id}] {stage} 실패: {type(error).__name__}: {error}", flush=True)
                raise
            finally:
                cleanup_cuda()
    comparison = build_comparison(reports)
    destination = project_path(EXPERIMENT_PARAMS["summary_file"])
    if destination.exists():
        if read_json(destination) != comparison:
            raise FileExistsError(f"기존 비교 결과와 다르므로 덮어쓰지 않습니다: {destination}")
    else:
        train_write_json(destination, comparison)
    display_comparison(comparison)
    print(f"비교 집계 저장: {destination}")
    return comparison


In [20]:
comparison = (
())


[seed_43_on] train
==((====))==  Unsloth 2026.10.9: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4070. Num GPUs = 1. Max memory: 11.994 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Unsloth 2026.10.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen3DecoderLayer).
trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,568 | Num Epochs = 3 | Total steps = 588
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)


...

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

평가 555건 / GPU: NVIDIA GeForce RTX 4070


Evaluation:  11%|█         | 61/555 [17:48<2:24:17, 17.52s/it]

[seed_44_off] 평가 실패: KeyboardInterrupt: 


KeyboardInterrupt: 

## 결과 해석과 재실행

시드별 on/off 차이, 클러스터 22 제외 차이와 클러스터별 차이를 함께 확인합니다. 평가 555건을 시드 수만큼 늘려 독립 표본으로 취급하지 않습니다. 통계적 유의성이나 운영 데이터 전반의 효과를 보장하지 않습니다.

학습 중 test.jsonl의 loss를 확인하는 기존 조건을 유지하며 최종 3에포크 모델로 생성 평가합니다. 평가 점수를 보고 에포크나 시드를 선택하지 않습니다. 증강에 따른 추가 작업과 토큰 노출량을 보상하지 않으므로 결합 자체의 효과와 노출량의 효과는 분리하지 않습니다.

완료 조건은 설정, 코드, 입력, 모델 및 이력 해시와 평가 대응을 확인한 뒤 건너뜁니다. 학습이 완료되고 평가가 없다면 평가부터 실행합니다. 미완료 학습은 자동으로 덮어쓰거나 재개하지 않으며 새 실행은 기존 자료를 보존한 새 실험 ID에서 진행합니다. 코드 해시는 저장된 코드 셀만 대상으로 하므로 결과 출력과 Markdown 수정은 해시에 영향을 주지 않습니다. 코드 수정은 완료 조건과 불일치로 처리합니다.
